# BSD10k Confidence Filtering v4: Score Calibration & Stacking

v3 결과에서 binary MLP는 F1을 조금 올렸지만, AUC-PR은 이전 5-class score보다 낮았다. 그래서 v4는 새 deep model을 바로 더 키우지 않고, 이미 있는 OOF score들을 제대로 쓰는 실험이다.

## 실험 목적

1. 이전 5-class score와 `P(c=4)+P(c=5)`도 threshold tuning한다.
2. v3 binary MLP probability와 5-class probability를 score-level ensemble한다.
3. OOF 기반 logistic stacker로 두 모델의 정보를 결합한다.
4. F1뿐 아니라 AUC-PR, precision@recall, retained ratio를 같이 본다.
5. 가장 좋은 score를 BSD35k-CS에 적용한다.

이 실험은 BSD35k-CS를 학습에 사용하지 않는다. BSD35k-CS에는 BSD10k OOF에서 결정한 threshold와 BSD10k 전체 score feature로 fit한 final stacker만 적용한다.


In [ ]:
from pathlib import Path
import json
import os

import numpy as np
import pandas as pd
try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ModuleNotFoundError:
    plt = None
    HAS_MATPLOTLIB = False
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

try:
    display
except NameError:
    def display(obj):
        print(obj)

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)

OUTPUT_DIR = ROOT / "outputs" / "confidence_filter_v4"
REPORT_DIR = OUTPUT_DIR / "reports"
PRED_DIR = OUTPUT_DIR / "predictions"
PLOT_DIR = OUTPUT_DIR / "plots"
for path in [REPORT_DIR, PRED_DIR, PLOT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

SEED = 42
DENSE_THRESHOLDS_01 = np.round(np.linspace(0.01, 0.99, 197), 4)
DENSE_THRESHOLDS_SCORE = np.round(np.linspace(1.0, 5.0, 401), 4)

V2_OOF = ROOT / "outputs" / "confidence_model_v2" / "predictions" / "BSD10k_oof_predicted_v2.csv"
V2_BSD35K = ROOT / "outputs" / "confidence_model_v2" / "predictions" / "BSD35k-CS_predicted_v2.csv"
V3_OOF = ROOT / "outputs" / "confidence_filter_v3" / "predictions" / "BSD10k_oof_binary_mlp.csv"
V3_BSD35K = ROOT / "outputs" / "confidence_filter_v3" / "predictions" / "BSD35k-CS_filter_predictions.csv"

for required in [V2_OOF, V2_BSD35K, V3_OOF, V3_BSD35K]:
    if not required.exists():
        raise FileNotFoundError(f"Missing required file: {required}")

print("root:", ROOT)
print("output:", OUTPUT_DIR)


In [ ]:
def safe_auc_pr(y_true, score):
    try:
        return float(average_precision_score(y_true, score))
    except ValueError:
        return float("nan")


def safe_auc_roc(y_true, score):
    try:
        return float(roc_auc_score(y_true, score))
    except ValueError:
        return float("nan")


def metrics_at_threshold(y_true, score, threshold):
    pred = (score >= threshold).astype(int)
    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, pred)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
        "auc_pr": safe_auc_pr(y_true, score),
        "auc_roc": safe_auc_roc(y_true, score),
    }


def threshold_sweep(y_true, score, thresholds):
    rows = []
    for threshold in thresholds:
        row = metrics_at_threshold(y_true, score, threshold)
        rows.append({
            "threshold": row["threshold"],
            "accuracy": row["accuracy"],
            "precision": row["precision"],
            "recall": row["recall"],
            "f1": row["f1"],
        })
    return pd.DataFrame(rows)


def pick_f1_optimal(y_true, score, thresholds):
    sweep = threshold_sweep(y_true, score, thresholds)
    return sweep.sort_values(["f1", "precision", "threshold"], ascending=[False, False, True]).iloc[0], sweep


def pick_precision_optimal_at_recall(y_true, score, thresholds, min_recall=0.7):
    sweep = threshold_sweep(y_true, score, thresholds)
    candidates = sweep[sweep["recall"] >= min_recall].copy()
    if len(candidates) == 0:
        return None, sweep
    return candidates.sort_values(["precision", "f1", "threshold"], ascending=[False, False, False]).iloc[0], sweep


def percentile_rank(x):
    s = pd.Series(x)
    return s.rank(method="average", pct=True).to_numpy(dtype=np.float32)


def markdown_table(df, float_digits=4):
    view = df.copy()
    for col in view.columns:
        if pd.api.types.is_float_dtype(view[col]):
            view[col] = view[col].map(lambda x: "" if pd.isna(x) else f"{x:.{float_digits}f}")
    view = view.fillna("")
    headers = [str(col) for col in view.columns]
    rows = [[str(value) for value in row] for row in view.to_numpy()]
    widths = [max([len(headers[i])] + [len(row[i]) for row in rows]) for i in range(len(headers))]
    header = "| " + " | ".join(headers[i].ljust(widths[i]) for i in range(len(headers))) + " |"
    sep = "| " + " | ".join("-" * widths[i] for i in range(len(headers))) + " |"
    body = ["| " + " | ".join(row[i].ljust(widths[i]) for i in range(len(headers))) + " |" for row in rows]
    return "\n".join([header, sep] + body)


In [ ]:
# Load and align BSD10k OOF predictions
v3_oof = pd.read_csv(V3_OOF)
v2_oof = pd.read_csv(V2_OOF)
v3_oof["sound_id"] = v3_oof["sound_id"].astype(str)
v2_oof["sound_id"] = v2_oof["sound_id"].astype(str)

base = v3_oof[["sound_id", "class", "class_top", "confidence", "target_high_confidence", "predicted_high_confidence_prob"]].copy()
cols_from_v2 = [
    "sound_id",
    "predicted_confidence_score",
    "prob_confidence_1",
    "prob_confidence_2",
    "prob_confidence_3",
    "prob_confidence_4",
    "prob_confidence_5",
]
df = base.merge(v2_oof[cols_from_v2].drop_duplicates("sound_id"), on="sound_id", how="inner")

df["binary_mlp_prob"] = df["predicted_high_confidence_prob"].astype(float)
df["fiveclass_score"] = df["predicted_confidence_score"].astype(float)
df["fiveclass_score_01"] = ((df["fiveclass_score"] - 1.0) / 4.0).clip(0, 1)
df["fiveclass_p45"] = df["prob_confidence_4"].astype(float) + df["prob_confidence_5"].astype(float)
df["fiveclass_margin_p45_minus_p3"] = df["fiveclass_p45"] - df["prob_confidence_3"].astype(float)
df["rank_average_binary_p45"] = (percentile_rank(df["binary_mlp_prob"]) + percentile_rank(df["fiveclass_p45"])) / 2.0
df["rank_average_binary_score"] = (percentile_rank(df["binary_mlp_prob"]) + percentile_rank(df["fiveclass_score"])) / 2.0

y = df["target_high_confidence"].to_numpy(dtype=int)
print("aligned rows:", len(df))
print("positive ratio:", y.mean())
display(df.head())


In [ ]:
# Single-score and rank-ensemble threshold tuning
score_specs = [
    ("binary_mlp_prob", "Binary MLP probability", DENSE_THRESHOLDS_01),
    ("fiveclass_p45", "5-class P4+P5", DENSE_THRESHOLDS_01),
    ("fiveclass_score", "5-class expected score", DENSE_THRESHOLDS_SCORE),
    ("fiveclass_score_01", "5-class expected score scaled 0-1", DENSE_THRESHOLDS_01),
    ("fiveclass_margin_p45_minus_p3", "5-class margin P45-P3", np.round(np.linspace(-1.0, 1.0, 401), 4)),
    ("rank_average_binary_p45", "Rank average: binary + P45", DENSE_THRESHOLDS_01),
    ("rank_average_binary_score", "Rank average: binary + expected score", DENSE_THRESHOLDS_01),
]

summary_rows = []
sweep_map = {}
for col, label, thresholds in score_specs:
    score = df[col].to_numpy(dtype=float)
    f1_best, sweep = pick_f1_optimal(y, score, thresholds)
    prec_best, _ = pick_precision_optimal_at_recall(y, score, thresholds, min_recall=0.7)
    sweep_map[label] = sweep
    summary_rows.append({
        "method": label,
        "selection": "F1-optimal",
        **metrics_at_threshold(y, score, float(f1_best["threshold"])),
    })
    if prec_best is not None:
        summary_rows.append({
            "method": label,
            "selection": "precision-optimal @ recall>=0.7",
            **metrics_at_threshold(y, score, float(prec_best["threshold"])),
        })

score_summary = pd.DataFrame(summary_rows)
score_summary.to_csv(REPORT_DIR / "v4_score_threshold_summary.csv", index=False)
display(score_summary.sort_values(["selection", "f1"], ascending=[True, False]))


In [ ]:
# OOF logistic stacker. This trains only on BSD10k OOF score features.
stack_features = [
    "binary_mlp_prob",
    "fiveclass_score_01",
    "fiveclass_p45",
    "prob_confidence_3",
    "prob_confidence_4",
    "prob_confidence_5",
    "fiveclass_margin_p45_minus_p3",
]
X = df[stack_features].to_numpy(dtype=np.float32)
stack_oof_prob = np.zeros(len(df), dtype=np.float32)
splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

for fold, (train_idx, valid_idx) in enumerate(splitter.split(X, y)):
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, solver="lbfgs"),
    )
    model.fit(X[train_idx], y[train_idx])
    stack_oof_prob[valid_idx] = model.predict_proba(X[valid_idx])[:, 1]

df["logistic_stacker_prob"] = stack_oof_prob
f1_best_stack, stack_sweep = pick_f1_optimal(y, stack_oof_prob, DENSE_THRESHOLDS_01)
prec_best_stack, _ = pick_precision_optimal_at_recall(y, stack_oof_prob, DENSE_THRESHOLDS_01, min_recall=0.7)

stack_rows = [
    {"method": "OOF logistic stacker", "selection": "F1-optimal", **metrics_at_threshold(y, stack_oof_prob, float(f1_best_stack["threshold"]))},
    {"method": "OOF logistic stacker", "selection": "precision-optimal @ recall>=0.7", **metrics_at_threshold(y, stack_oof_prob, float(prec_best_stack["threshold"]))},
]
stack_summary = pd.DataFrame(stack_rows)

all_summary = pd.concat([score_summary, stack_summary], ignore_index=True)
all_summary.to_csv(REPORT_DIR / "v4_all_method_summary.csv", index=False)
df.to_csv(PRED_DIR / "BSD10k_oof_v4_scores.csv", index=False)

display(all_summary.sort_values(["selection", "f1", "auc_pr"], ascending=[True, False, False]))


In [ ]:
# Curves for the most relevant candidates
plot_candidates = {
    "Binary MLP": df["binary_mlp_prob"].to_numpy(dtype=float),
    "5-class P4+P5": df["fiveclass_p45"].to_numpy(dtype=float),
    "5-class score scaled": df["fiveclass_score_01"].to_numpy(dtype=float),
    "Rank avg binary+P45": df["rank_average_binary_p45"].to_numpy(dtype=float),
    "Logistic stacker": df["logistic_stacker_prob"].to_numpy(dtype=float),
}

if HAS_MATPLOTLIB:
    plt.figure(figsize=(7, 6))
    for name, score in plot_candidates.items():
        precision, recall, _ = precision_recall_curve(y, score)
        plt.plot(recall, precision, label=f"{name} AP={safe_auc_pr(y, score):.4f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("v4 PR curves on BSD10k OOF")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "v4_pr_curves.png", dpi=160)
    plt.show()

    plt.figure(figsize=(7, 5))
    for name, score in plot_candidates.items():
        best, sweep = pick_f1_optimal(y, score, DENSE_THRESHOLDS_01)
        if score.max() > 1.0 or score.min() < 0.0:
            continue
        plt.plot(sweep["threshold"], sweep["f1"], label=name)
    plt.xlabel("Threshold")
    plt.ylabel("F1")
    plt.title("v4 F1 by threshold")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "v4_threshold_f1.png", dpi=160)
    plt.show()
else:
    print("matplotlib is not installed; skipping plots.")


In [ ]:
# Select best F1 method and apply to BSD35k-CS
f1_candidates = all_summary[all_summary["selection"] == "F1-optimal"].copy()
best_row = f1_candidates.sort_values(["f1", "auc_pr", "precision"], ascending=[False, False, False]).iloc[0]
best_method = str(best_row["method"])
best_threshold = float(best_row["threshold"])
print("best F1 method:", best_method, "threshold:", best_threshold)

v3_bsd35k = pd.read_csv(V3_BSD35K)
v2_bsd35k = pd.read_csv(V2_BSD35K)
v3_bsd35k["sound_id"] = v3_bsd35k["sound_id"].astype(str)
v2_bsd35k["sound_id"] = v2_bsd35k["sound_id"].astype(str)

base35 = v3_bsd35k.copy()
v2_cols_35 = [
    "sound_id",
    "predicted_confidence_score",
    "prob_confidence_1",
    "prob_confidence_2",
    "prob_confidence_3",
    "prob_confidence_4",
    "prob_confidence_5",
]
bsd35 = base35.merge(v2_bsd35k[v2_cols_35].drop_duplicates("sound_id"), on="sound_id", how="inner", suffixes=("", "_v2"))

bsd35["binary_mlp_prob"] = bsd35["predicted_high_confidence_prob"].astype(float)
bsd35["fiveclass_score"] = bsd35["predicted_confidence_score"].astype(float)
bsd35["fiveclass_score_01"] = ((bsd35["fiveclass_score"] - 1.0) / 4.0).clip(0, 1)
bsd35["fiveclass_p45"] = bsd35["prob_confidence_4"].astype(float) + bsd35["prob_confidence_5"].astype(float)
bsd35["fiveclass_margin_p45_minus_p3"] = bsd35["fiveclass_p45"] - bsd35["prob_confidence_3"].astype(float)
bsd35["rank_average_binary_p45"] = (percentile_rank(bsd35["binary_mlp_prob"]) + percentile_rank(bsd35["fiveclass_p45"])) / 2.0
bsd35["rank_average_binary_score"] = (percentile_rank(bsd35["binary_mlp_prob"]) + percentile_rank(bsd35["fiveclass_score"])) / 2.0

final_stacker = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, solver="lbfgs"),
)
final_stacker.fit(X, y)
bsd35["logistic_stacker_prob"] = final_stacker.predict_proba(bsd35[stack_features].to_numpy(dtype=np.float32))[:, 1]

method_to_col = {
    "Binary MLP probability": "binary_mlp_prob",
    "5-class P4+P5": "fiveclass_p45",
    "5-class expected score": "fiveclass_score",
    "5-class expected score scaled 0-1": "fiveclass_score_01",
    "5-class margin P45-P3": "fiveclass_margin_p45_minus_p3",
    "Rank average: binary + P45": "rank_average_binary_p45",
    "Rank average: binary + expected score": "rank_average_binary_score",
    "OOF logistic stacker": "logistic_stacker_prob",
}
best_col = method_to_col[best_method]
bsd35["v4_filter_score"] = bsd35[best_col].astype(float)
bsd35["v4_predicted_high_confidence"] = (bsd35["v4_filter_score"] >= best_threshold).astype(int)
bsd35["v4_best_method"] = best_method
bsd35["v4_best_threshold"] = best_threshold

scenario_rows = []
if best_col == "fiveclass_score":
    scenario_thresholds = sorted(set([3.0, 3.25, 3.5, 3.75, 4.0, best_threshold]))
elif best_col == "fiveclass_margin_p45_minus_p3":
    scenario_thresholds = sorted(set([-0.2, 0.0, 0.2, 0.4, 0.6, best_threshold]))
else:
    scenario_thresholds = sorted(set([0.5, 0.6, 0.7, 0.8, 0.9, best_threshold]))
for threshold in scenario_thresholds:
    kept = int((bsd35["v4_filter_score"] >= threshold).sum())
    oof_metric = metrics_at_threshold(y, df[method_to_col[best_method]].to_numpy(dtype=float), threshold)
    scenario_rows.append({
        "method": best_method,
        "threshold": threshold,
        "retained_samples": kept,
        "retained_ratio": kept / len(bsd35),
        "expected_precision_from_oof": oof_metric["precision"],
        "expected_recall_from_oof": oof_metric["recall"],
        "expected_f1_from_oof": oof_metric["f1"],
    })
scenario_df = pd.DataFrame(scenario_rows)
scenario_df.to_csv(REPORT_DIR / "v4_bsd35k_filtering_scenarios.csv", index=False)

export_cols = [
    "sample_id",
    "sound_id",
    "class",
    "class_idx",
    "class_top",
    "uploader",
    "title",
    "tags",
    "description",
    "binary_mlp_prob",
    "fiveclass_score",
    "fiveclass_p45",
    "rank_average_binary_p45",
    "logistic_stacker_prob",
    "v4_filter_score",
    "v4_predicted_high_confidence",
    "v4_best_method",
    "v4_best_threshold",
]
export_cols = [col for col in export_cols if col in bsd35.columns]
bsd35[export_cols].to_csv(PRED_DIR / "BSD35k-CS_filter_predictions_v4.csv", index=False)
display(scenario_df)
print("saved:", PRED_DIR / "BSD35k-CS_filter_predictions_v4.csv")


In [ ]:
# Write v4 report
top_f1 = all_summary[all_summary["selection"] == "F1-optimal"].sort_values(["f1", "auc_pr"], ascending=[False, False]).head(10)
top_precision = all_summary[all_summary["selection"] == "precision-optimal @ recall>=0.7"].sort_values(["precision", "f1"], ascending=[False, False]).head(10)
plot_lines = [
    "- `outputs/confidence_filter_v4/plots/v4_pr_curves.png`",
    "- `outputs/confidence_filter_v4/plots/v4_threshold_f1.png`",
] if HAS_MATPLOTLIB else ["- matplotlib이 설치되어 있지 않아 현재 실행에서는 plot 생성을 건너뜀"]

report = [
    "# BSD10k Confidence Filtering v4 보고서",
    "",
    "## 1. 실험 목적",
    "",
    "v3 binary MLP는 F1은 조금 개선했지만 AUC-PR은 이전 5-class score보다 낮았다. v4는 새 deep model을 키우는 대신, 이미 있는 score를 threshold tuning, rank ensemble, logistic stacker로 조합한다.",
    "",
    "## 2. 사용한 score",
    "",
    "- Binary MLP probability: v3에서 학습한 binary probability",
    "- 5-class expected score: 이전 `stage2_baseline_emd`의 expected confidence score",
    "- 5-class P4+P5: confidence 4 또는 5일 확률 합",
    "- Rank average: 서로 scale이 다른 score의 순위 평균",
    "- OOF logistic stacker: BSD10k OOF score feature만 사용한 logistic regression",
    "",
    "## 3. F1-optimal 비교",
    "",
    markdown_table(top_f1),
    "",
    "## 4. Precision-oriented 비교",
    "",
    markdown_table(top_precision),
    "",
    "## 5. 선택된 BSD35k-CS 적용 방식",
    "",
    f"- best method: `{best_method}`",
    f"- best threshold: {best_threshold:.4f}",
    f"- prediction file: `outputs/confidence_filter_v4/predictions/BSD35k-CS_filter_predictions_v4.csv`",
    "",
    "## 6. BSD35k-CS filtering scenarios",
    "",
    markdown_table(scenario_df),
    "",
    "## 7. 참고 plot",
    "",
    *plot_lines,
]

report_text = "\n".join(report)
(REPORT_DIR / "confidence_filter_v4_report_ko.md").write_text(report_text, encoding="utf-8")
(ROOT / "confidence_filter_v4_report_ko.md").write_text(report_text, encoding="utf-8")

summary = {
    "best_method": best_method,
    "best_threshold": best_threshold,
    "stack_features": stack_features,
    "prediction_file": str(PRED_DIR / "BSD35k-CS_filter_predictions_v4.csv"),
}
(REPORT_DIR / "confidence_filter_v4_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print("report saved:", REPORT_DIR / "confidence_filter_v4_report_ko.md")
